![CREWS UFFFS Nowcast](logo.png)

# Tutorial 2: Create a GSMaP precipitation nowcast with pySTEPS

In this tutorial you will turn the processed GSMaP rainfall fields from Tutorial 1 into a short-term **probabilistic nowcast** using the pySTEPS library. You will:

- Configure the nowcast settings (input window, lead time, ensemble size, domain).
- Load the processed GSMaP NetCDF and select the most recent input fields.
- Build a simple **persistence baseline** for context.
- Run a **STEPS ensemble nowcast** and a **deterministic extrapolation** diagnostic.
- Visualise the estimated motion vectors, ensemble mean and ensemble maximum.
- Export the nowcast to NetCDF for re-use in Tutorial 3.

Tutorial 3 picks up where this notebook ends and covers ensemble products, city-level time series, gauge validation, and operational outputs.

# Step 1: Import the necessary libraries

We use `xarray` for gridded rainfall data, `matplotlib` for plotting, and `pySTEPS` for optical-flow nowcasting.

If pySTEPS fails to import, check the Pixi environment. In particular, older pySTEPS versions may still depend on `pkg_resources`, which requires a compatible `setuptools` version.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import warnings

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import imageio.v2 as imageio
import rioxarray  # noqa: F401. Required for GeoTIFF export with xarray/rioxarray.

warnings.filterwarnings("ignore", category=RuntimeWarning)

print("Libraries imported.")

# Step 2: Define the nowcast settings

The settings below should match the event downloaded in Tutorial 1.

The most important nowcast settings are:

- `n_input_files`: how many observed rainfall fields are used to estimate motion;
- `n_lead_times`: how many forecast steps are produced;
- `timestep_minutes`: the time interval of the input data;
- `n_ensemble_members`: the number of stochastic ensemble members;
- `rain_threshold_mm_h`: the rainfall threshold used to separate rain from no rain.

For GSMaP hourly data, each lead time step corresponds to one hour when `timestep_minutes = 60`.

In [ ]:
settings = {
    # ---------------------------------------------------------------------
    # General workshop metadata
    # ---------------------------------------------------------------------
    "project_name": "CREWS_Lao_PDR_training",
    "country_or_region": "Southeast Asia example domain",
    "event_name": "Example heavy rainfall event",

    # ---------------------------------------------------------------------
    # Google Earth Engine settings
    # ---------------------------------------------------------------------
    # For many users, Earth Engine now asks for a Cloud project during initialization.
    # Put your project ID here if you have one, for example: "my-gee-project-123".
    # Leave as None if your existing local Earth Engine credentials initialize without it.
    "gee_project": None,

    # GSMaP operational rainfall-rate collection in Google Earth Engine.
    "ee_collection": "JAXA/GPM_L3/GSMaP/v7/operational",

    # Rainfall-rate band.
    # "hourlyPrecipRate" is the near-real-time hourly rainfall rate.
    # "hourlyPrecipRateGC" is gauge-corrected where available, but may have more latency.
    "band": "hourlyPrecipRate",

    # ---------------------------------------------------------------------
    # Spatial and temporal user inputs
    # ---------------------------------------------------------------------
    # Bounding box: [min_lon, min_lat, max_lon, max_lat]
    "bbox": [95.0, -12.0, 155.0, 20.0],

    # City-focused zoom plots.
    # The model/download domain can remain large, while the plots below can zoom in
    # around locations that are useful for discussion during the workshop.
    "city_zoom_enabled": True,
    "city_zoom_buffer_degrees": 2.0,
    "city_zoom_locations": [
        {
            "name": "Phnom Penh",
            "country": "Cambodia",
            "lat": 11.5564,
            "lon": 104.9282,
            "notes": "City-centre reference point; adjust if your project uses a specific station or district.",
        },
        {
            "name": "Vientiane",
            "country": "Lao PDR",
            "lat": 17.9757,
            "lon": 102.6331,
            "notes": "City-centre reference point; adjust if your project uses a specific station or district.",
        },
    ],

    # ---------------------------------------------------------------------
    # Plotting options
    # ---------------------------------------------------------------------
    # If True, 0 mm h-1 rainfall values are converted to NaN only while plotting.
    # This makes dry pixels transparent and allows the basemap to remain visible.
    "plot_zero_as_nan": True,

    # If True, the map plotting helper tries to add an OpenStreetMap basemap.
    # This requires internet access and the optional contextily package.
    "plot_use_osm_basemap": True,
    "plot_basemap_alpha": 0.65,

    # Download window in UTC. Use full hours for GSMaP.
    "start_time": "2025-10-01T00:00:00Z",
    "end_time": "2025-10-01T12:00:00Z",

    # Last observed timestep to use for the nowcast. None means: use the latest available field.
    "nowcast_reference_time": None,

    # Approximate GSMaP native resolution. Earth Engine downloads use metres for scale.
    # 0.1 degrees is approximately 11132 m at the equator.
    "download_scale_m": 11132,

    # ---------------------------------------------------------------------
    # Nowcast settings
    # ---------------------------------------------------------------------
    # GSMaP is hourly, so each lead time is one hour.
    "n_input_files": 6,
    "n_lead_times": 6,
    "timestep_minutes": 60,

    # Rainfall values below this threshold are treated as no rain for the nowcast.
    # This reduces noise in the motion estimation.
    "rain_threshold_mm_h": 0.1,

    # Prefer probabilistic STEPS if it works. Otherwise, fall back to deterministic extrapolation.
    "use_steps_if_available": True,
    "n_ensemble_members": 8,

    # Approximate grid spacing in km used by STEPS stochastic perturbations.
    # GSMaP 0.1 degree is roughly 11 km at the equator, but the zonal spacing varies with latitude.
    "km_per_pixel": 10.0,

    # Reproducibility seed for the stochastic nowcast.
    "random_seed": 42,
}

settings


# Step 3: Define the data folder structure

The processed NetCDF file from Tutorial 1 is read from `gsmap_data/processed_netcdf`.

Nowcast products, CSV tables and figures are saved in the `nowcast` and `figures` folders.

In [ ]:
# Create folders for the processed input data, nowcast output, figures, and tables.
data_folder = Path("./gsmap_data")
processed_folder = data_folder / "processed_netcdf"
nowcast_folder = data_folder / "nowcast"
figures_folder = data_folder / "figures"

for folder in [processed_folder, nowcast_folder, figures_folder]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Processed NetCDF folder: {processed_folder.resolve()}")
print(f"Nowcast output folder: {nowcast_folder.resolve()}")
print(f"Figures folder: {figures_folder.resolve()}")

## Optional: OpenStreetMap basemaps and transparent dry pixels

Rainfall maps are easier to interpret when the surrounding geography is visible. In this notebook, the plotting helper can place an OpenStreetMap basemap below the rainfall layer.

The helper also converts rainfall values of 0 mm h-1 to `NaN` for plotting. This does not change the data itself. It only makes dry pixels transparent in the figures, so that coastlines, rivers, borders and cities remain visible below the rainfall.

The basemap requires the optional package `contextily`. If it is not installed, the notebook will still run; it will simply print a message and continue without the basemap. To enable it in the Pixi environment, run this once in a terminal:

```powershell
pixi add contextily xyzservices
```


In [ ]:
def prepare_field_for_plot(da, zero_as_nan=None):
    """Return a copy of a DataArray that is convenient for plotting.

    For rainfall maps, values of exactly 0 often dominate the figure even though
    they simply mean 'no rain'. During a workshop, it is usually clearer to make
    these dry pixels transparent and let the basemap show through.

    This function only affects the plotted field. It does not modify the original
    xarray object or the values saved in NetCDF/GeoTIFF outputs.
    """
    if zero_as_nan is None:
        zero_as_nan = settings.get("plot_zero_as_nan", True)

    field = da.squeeze()

    if zero_as_nan:
        field = field.where(field != 0)

    return field


def get_transparent_colormap(cmap_name="Blues"):
    """Return a matplotlib colormap where NaN values are transparent."""
    cmap = plt.get_cmap(cmap_name).copy()
    cmap.set_bad((1, 1, 1, 0))
    return cmap


def add_openstreetmap_basemap(ax, enabled=None, alpha=None):
    """Add an OpenStreetMap basemap to a lon/lat matplotlib axis if possible.

    The rainfall data are plotted in EPSG:4326 longitude/latitude coordinates.
    `contextily` can fetch OSM tiles and reproject them to the same CRS. This
    requires internet access and the optional `contextily` package.
    """
    if enabled is None:
        enabled = settings.get("plot_use_osm_basemap", True)
    if alpha is None:
        alpha = settings.get("plot_basemap_alpha", 0.65)

    if not enabled:
        return

    xlim = ax.get_xlim()
    ylim = ax.get_ylim()

    try:
        import contextily as ctx
    except ImportError:
        print(
            "OpenStreetMap basemap skipped: contextily is not installed. "
            "Install it with: pixi add contextily xyzservices"
        )
        return

    try:
        ctx.add_basemap(
            ax,
            crs="EPSG:4326",
            source=ctx.providers.OpenStreetMap.Mapnik,
            alpha=alpha,
            attribution_size=6,
            reset_extent=False,
        )
        ax.set_xlim(xlim)
        ax.set_ylim(ylim)
    except Exception as exc:
        print(f"OpenStreetMap basemap skipped: {exc}")


def plot_gridded_map(
    da,
    title,
    cmap="Blues",
    vmin=0,
    vmax=None,
    zero_as_nan=None,
    add_basemap=None,
    city=None,
    figure_size=(10, 5),
    cbar_label=None,
    output_path=None,
):
    """Plot a lat/lon DataArray with optional OSM basemap and transparent dry pixels."""
    field = prepare_field_for_plot(da, zero_as_nan=zero_as_nan)
    cmap_obj = get_transparent_colormap(cmap)

    fig, ax = plt.subplots(figsize=figure_size)

    # Set the map extent before adding the basemap. Contextily uses the current
    # axis limits to determine which map tiles to fetch.
    ax.set_xlim(float(field.lon.min()), float(field.lon.max()))
    ax.set_ylim(float(field.lat.min()), float(field.lat.max()))

    if add_basemap is None:
        add_basemap = settings.get("plot_use_osm_basemap", True)
    add_openstreetmap_basemap(ax, enabled=add_basemap)

    if cbar_label is None:
        cbar_label = field.attrs.get("units", "")

    field.plot.pcolormesh(
        ax=ax,
        x="lon",
        y="lat",
        cmap=cmap_obj,
        vmin=vmin,
        vmax=vmax,
        add_colorbar=True,
        cbar_kwargs={"label": cbar_label},
        zorder=2,
    )

    if city is not None:
        ax.scatter(
            float(city["lon"]),
            float(city["lat"]),
            marker="x",
            s=80,
            label=city["name"],
            zorder=3,
        )
        ax.legend()

    ax.set_title(title)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    fig.tight_layout()

    if output_path is not None:
        fig.savefig(output_path, dpi=140, bbox_inches="tight")
        print(f"Saved: {output_path}")

    plt.show()
    return fig, ax


# Step 4: Load the processed GSMaP rainfall data

The notebook searches for processed GSMaP NetCDF files and opens the latest one.

If you want to use a specific file, replace `processed_file = processed_files[-1]` with the exact path to your NetCDF file.

In [ ]:
# Search for processed GSMaP NetCDF files from Tutorial 1.
processed_files = sorted(processed_folder.glob("GSMaP_*.nc"))

if not processed_files:
    raise FileNotFoundError(
        "No processed GSMaP NetCDF files were found. "
        "Run Tutorial 1 first, or place a processed GSMaP NetCDF file in gsmap_data/processed_netcdf."
    )

# Use the latest file by filename. You can also replace this with a specific file path.
processed_file = processed_files[-1]
print(f"Opening processed GSMaP file: {processed_file}")

gsmap_ds = xr.open_dataset(processed_file)
gsmap_ds = gsmap_ds.assign_coords(time=pd.to_datetime(gsmap_ds.time.values)).sortby("time")

gsmap_ds

# Step 5: Define city zoom functions

The full domain is useful for motion estimation, but city zooms are easier to interpret.

We reuse the same city locations defined in the settings: Phnom Penh and Vientiane by default.

In [ ]:
def get_city_zoom_table(settings):
    """Return city zoom settings as a small table.

    The table is useful in a workshop because it makes all selected cities and
    zoom boxes explicit. Participants can edit the latitude, longitude, or buffer
    and immediately re-run the plots.
    """
    records = []

    for city in settings.get("city_zoom_locations", []):
        buffer_deg = float(settings.get("city_zoom_buffer_degrees", 2.0))
        lat = float(city["lat"])
        lon = float(city["lon"])

        records.append({
            "name": city["name"],
            "country": city.get("country", ""),
            "lat": lat,
            "lon": lon,
            "buffer_degrees": buffer_deg,
            "min_lon": lon - buffer_deg,
            "max_lon": lon + buffer_deg,
            "min_lat": lat - buffer_deg,
            "max_lat": lat + buffer_deg,
            "notes": city.get("notes", ""),
        })

    return pd.DataFrame(records)


def subset_dataarray_around_city(da, city, buffer_degrees):
    """Subset a DataArray to a square lon/lat box around one city.

    The function assumes that coordinates are named ``lat`` and ``lon``.
    It works with latitude in ascending or descending order, although earlier
    in this notebook we sort latitude ascending for convenience.
    """
    lat = float(city["lat"])
    lon = float(city["lon"])
    buffer_degrees = float(buffer_degrees)

    min_lon = lon - buffer_degrees
    max_lon = lon + buffer_degrees
    min_lat = lat - buffer_degrees
    max_lat = lat + buffer_degrees

    if da.lat.values[0] <= da.lat.values[-1]:
        lat_slice = slice(min_lat, max_lat)
    else:
        lat_slice = slice(max_lat, min_lat)

    return da.sel(lon=slice(min_lon, max_lon), lat=lat_slice)


def plot_city_zoom_field(da, settings, title_prefix, file_prefix=None):
    """Plot one rainfall field around each configured city.

    The city plots use the same map helper as the full-domain plots. This means
    that 0 values are transparent by default and an OpenStreetMap basemap is shown
    when `contextily` is installed.
    """
    if not settings.get("city_zoom_enabled", True):
        print("City zooms are disabled. Set settings['city_zoom_enabled'] = True to enable them.")
        return

    buffer_degrees = settings.get("city_zoom_buffer_degrees", 2.0)

    for city in settings.get("city_zoom_locations", []):
        city_da = subset_dataarray_around_city(da, city, buffer_degrees)

        output_path = None
        if file_prefix is not None:
            safe_name = city["name"].lower().replace(" ", "_")
            output_path = figures_folder / f"{file_prefix}_{safe_name}.png"

        plot_gridded_map(
            city_da,
            title=(
                f"{title_prefix}\n"
                f"{city['name']} zoom, +/- {buffer_degrees} degrees"
            ),
            cmap="Blues",
            vmin=0,
            zero_as_nan=settings.get("plot_zero_as_nan", True),
            city=city,
            figure_size=(8, 6),
            cbar_label=city_da.attrs.get("units", "mm h-1"),
            output_path=output_path,
        )


city_zoom_table = get_city_zoom_table(settings)
city_zoom_table


# Step 6: Select the input fields for the nowcast

pySTEPS needs a short sequence of recent rainfall fields. These fields are used to estimate how rainfall patterns are moving.

For example, if `n_input_files = 6` and the timestep is hourly, the nowcast uses the last six hours of rainfall observations.

If `settings["nowcast_reference_time"]` is `None`, the latest available timestep is used as the last observation.

In [ ]:
def select_nowcast_input(ds, n_input_files, reference_time=None):
    """Select a consecutive block of input fields ending at reference_time.

    If reference_time is None, the latest available timestep is used.
    """
    ds = ds.sortby("time")

    if reference_time is None:
        selected = ds.isel(time=slice(-n_input_files, None))
    else:
        reference_time = pd.to_datetime(reference_time, utc=True).tz_convert(None)
        selected = ds.sel(time=slice(None, reference_time)).isel(time=slice(-n_input_files, None))

    if selected.sizes["time"] < n_input_files:
        raise ValueError(
            f"Only {selected.sizes['time']} input fields available, "
            f"but {n_input_files} are required. Download a longer period or reduce n_input_files."
        )

    return selected


input_ds = select_nowcast_input(
    gsmap_ds,
    settings["n_input_files"],
    reference_time=settings["nowcast_reference_time"],
)

print("Selected input times for motion estimation:")
for value in input_ds.time.values:
    print(" -", pd.to_datetime(value), "UTC")

input_ds


# Step 7: Prepare the rainfall input for pySTEPS

pySTEPS expects a numerical array with dimensions:

```text
time, latitude, longitude
```

Very small rainfall values are set to zero. This reduces noise and helps the motion-estimation algorithm focus on meaningful rainfall structures.

In [ ]:
precip_da = input_ds["precipitation_rate"]

# Convert to a dense NumPy array. Values below the threshold are set to zero.
precip = precip_da.fillna(0).values.astype(np.float32)
precip = np.where(precip >= settings["rain_threshold_mm_h"], precip, 0.0)

print("Input array shape (time, lat, lon):", precip.shape)
print("Minimum rainfall rate:", float(np.nanmin(precip)), "mm h-1")
print("Maximum rainfall rate:", float(np.nanmax(precip)), "mm h-1")
print("Number of wet pixels:", int(np.sum(precip >= settings["rain_threshold_mm_h"])))


# Step 8: Create a simple persistence baseline

A persistence nowcast assumes that the latest observed rainfall field remains unchanged during the forecast period.

This is a very simple forecast. It does not move rainfall systems and it does not represent uncertainty. However, it is a useful baseline. A more advanced nowcast should normally provide more information than persistence.

In [ ]:
def create_persistence_nowcast(input_ds, settings):
    """Create a simple persistence baseline from the latest observed rainfall field."""
    latest = input_ds["precipitation_rate"].isel(time=-1).fillna(0)

    lead_steps = np.arange(1, settings["n_lead_times"] + 1)
    last_input_time = pd.to_datetime(input_ds.time.values[-1]).tz_localize(None)

    valid_times = np.array(
        [
            last_input_time + pd.Timedelta(minutes=settings["timestep_minutes"] * int(step))
            for step in lead_steps
        ],
        dtype="datetime64[ns]",
    )

    persistence = xr.concat(
        [latest for _ in lead_steps],
        dim="time",
    )

    persistence = persistence.assign_coords(
        time=valid_times,
        lead_time=("time", lead_steps),
    )

    persistence.name = "precipitation_rate_persistence"
    persistence.attrs.update(
        {
            "units": "mm h-1",
            "long_name": "Persistence nowcast precipitation rate",
            "description": "Latest observed rainfall field repeated for all forecast lead times.",
        }
    )

    return persistence


persistence_nowcast = create_persistence_nowcast(input_ds, settings)

first_persistence = persistence_nowcast.isel(time=0)
plot_gridded_map(
    first_persistence,
    title=(
        "Persistence baseline | "
        f"lead time {int(first_persistence.lead_time.values)} h | "
        f"valid {pd.to_datetime(first_persistence.time.values)} UTC"
    ),
    cmap="Blues",
    vmin=0,
    zero_as_nan=settings.get("plot_zero_as_nan", True),
    cbar_label="mm h-1",
)


## Exercise 1: Understand persistence

Look at the persistence plot and discuss:

1. What assumption is persistence making?
2. When could persistence be a reasonable approximation?
3. When would persistence fail for a moving convective system?
4. Why is it useful to compare pySTEPS against this simple baseline?

# Step 9: What pySTEPS does in this notebook

pySTEPS is an open-source Python library for precipitation nowcasting.

In this notebook, pySTEPS is used for two main tasks:

1. **Motion estimation**: estimate how rainfall features move between recent observations.
2. **Nowcasting**: move the rainfall field forward in time and optionally create ensemble members.

The basic workflow is:

```text
Observed rainfall sequence
        |
        v
Rain/no-rain threshold and transformation
        |
        v
Optical-flow motion estimation
        |
        v
Rainfall extrapolation
        |
        v
Optional stochastic perturbations
        |
        v
Ensemble nowcast, maps, probabilities, and time series
```

The motion vectors are estimated from the observed rainfall sequence. These vectors are then used to extrapolate rainfall into the next forecast hours.

Important limitations:

- pySTEPS mainly extrapolates existing rainfall structures;
- it does not fully predict new storm initiation;
- nowcast skill usually decreases quickly with lead time;
- satellite rainfall uncertainty remains part of the nowcast uncertainty.

# Step 10: Run the pySTEPS nowcast

This cell first estimates the motion field and then attempts to run the STEPS ensemble nowcast.

If STEPS fails, the notebook falls back to deterministic extrapolation. This fallback is useful in a training environment because participants can still continue the exercise even when the stochastic nowcast is not available.

In [ ]:
def run_pysteps_nowcast(precip, settings):
    """Run a pySTEPS nowcast, preferably STEPS, with deterministic fallback.

    Returns
    -------
    forecast : np.ndarray
        Nowcast rainfall rate in mm h-1. Shape is either
        (member, lead_time, lat, lon) or (1, lead_time, lat, lon).
    velocity : np.ndarray
        Motion field with shape (2, lat, lon). The first component is x-motion,
        the second component is y-motion, in pixels per timestep.
    nowcast_method : str
        Name of the method that was successfully used.
    precip_log : np.ndarray
        Transformed rainfall input used internally by pySTEPS.
    metadata : dict
        Metadata needed for inverse transformation from dB back to rainfall rate.
    """
    try:
        from pysteps import motion, nowcasts
        from pysteps.utils import transformation
    except Exception as exc:
        raise ImportError(
            "pySTEPS is not installed or could not be imported. "
            "In the Pixi environment, check that pysteps and setuptools<81 are installed."
        ) from exc

    precip_for_motion = precip.copy()
    precip_for_motion[precip_for_motion < settings["rain_threshold_mm_h"]] = 0.0

    # Log/dB transformation. Dry values are represented by zerovalue.
    precip_log, metadata = transformation.dB_transform(
        precip_for_motion,
        threshold=settings["rain_threshold_mm_h"],
        zerovalue=-15.0,
    )

    # Estimate optical-flow motion. "LK" is the Lucas-Kanade method in pySTEPS.
    oflow_method = motion.get_method("LK")
    velocity = oflow_method(precip_log)
    print("Estimated motion field shape:", velocity.shape)
    print("Velocity units: pixels per timestep")

    if settings["use_steps_if_available"]:
        try:
            steps_method = nowcasts.get_method("steps")
            forecast_log = steps_method(
                precip_log,
                velocity,
                settings["n_lead_times"],
                n_ens_members=settings["n_ensemble_members"],
                n_cascade_levels=6,
                precip_thr=settings["rain_threshold_mm_h"],
                kmperpixel=settings["km_per_pixel"],
                timestep=settings["timestep_minutes"],
                noise_method="nonparametric",
                vel_pert_method="bps",
                mask_method="incremental",
                probmatching_method="cdf",
                seed=settings["random_seed"],
            )

            forecast = transformation.dB_transform(
                forecast_log,
                threshold=settings["rain_threshold_mm_h"],
                zerovalue=-15.0,
                inverse=True,
                metadata=metadata,
            )[0]

            forecast = np.where(np.isfinite(forecast), forecast, 0.0)
            forecast = np.maximum(forecast, 0.0)
            return forecast.astype(np.float32), velocity, "STEPS", precip_log, metadata

        except Exception as exc:
            print("STEPS failed; falling back to deterministic extrapolation.")
            print(f"Reason: {exc}")

    extrapolate = nowcasts.get_method("extrapolation")
    forecast = extrapolate(
        precip[-1, :, :],
        velocity,
        settings["n_lead_times"],
    )
    forecast = np.where(np.isfinite(forecast), forecast, 0.0)
    forecast = np.maximum(forecast, 0.0)
    return forecast[np.newaxis, ...].astype(np.float32), velocity, "extrapolation", precip_log, metadata


forecast_array, velocity, nowcast_method, precip_log, pysteps_metadata = run_pysteps_nowcast(precip, settings)
print("Nowcast method:", nowcast_method)
print("Forecast array shape:", forecast_array.shape)


# Step 11: Visualize the estimated motion vectors

The arrows show the estimated movement of rainfall features.

The units are pixels per timestep. With hourly GSMaP input, one timestep is one hour.

Use these plots as a diagnostic. If the arrows look unrealistic, the nowcast should be interpreted with caution.

In [ ]:
def plot_motion_vectors(input_ds, velocity, skip=12):
    """Plot the latest rainfall field with pySTEPS optical-flow motion vectors."""
    latest_rain = input_ds["precipitation_rate"].isel(time=-1)

    lon = input_ds.lon.values
    lat = input_ds.lat.values
    lon2d, lat2d = np.meshgrid(lon, lat)

    vx = velocity[0]
    vy = velocity[1]

    fig, ax = plot_gridded_map(
        latest_rain,
        title=(
            "Estimated rainfall motion field from pySTEPS optical flow\n"
            f"latest input: {pd.to_datetime(latest_rain.time.values)} UTC"
        ),
        cmap="Blues",
        vmin=0,
        zero_as_nan=settings.get("plot_zero_as_nan", True),
        figure_size=(11, 6),
        cbar_label="mm h-1",
    )

    ax.quiver(
        lon2d[::skip, ::skip],
        lat2d[::skip, ::skip],
        vx[::skip, ::skip],
        vy[::skip, ::skip],
        angles="xy",
        scale_units="xy",
        scale=1,
        width=0.002,
        zorder=3,
    )
    fig.canvas.draw_idle()


def plot_motion_vectors_city_zoom(input_ds, velocity, settings, skip=4):
    """Plot pySTEPS motion vectors in zoom boxes around configured cities.

    This is usually easier to explain in a workshop than the full-domain vector
    plot. The arrows show the estimated movement of rainfall features in grid
    cells per timestep. With hourly GSMaP input, one timestep is one hour.
    """
    if not settings.get("city_zoom_enabled", True):
        print("City zooms are disabled. Set settings['city_zoom_enabled'] = True to enable them.")
        return

    latest_rain = input_ds["precipitation_rate"].isel(time=-1)

    vx_da = xr.DataArray(
        velocity[0],
        coords={"lat": input_ds.lat.values, "lon": input_ds.lon.values},
        dims=("lat", "lon"),
        name="x_motion_pixels_per_timestep",
    )
    vy_da = xr.DataArray(
        velocity[1],
        coords={"lat": input_ds.lat.values, "lon": input_ds.lon.values},
        dims=("lat", "lon"),
        name="y_motion_pixels_per_timestep",
    )

    buffer_degrees = settings.get("city_zoom_buffer_degrees", 2.0)

    for city in settings.get("city_zoom_locations", []):
        rain_zoom = subset_dataarray_around_city(latest_rain, city, buffer_degrees)
        vx_zoom = subset_dataarray_around_city(vx_da, city, buffer_degrees)
        vy_zoom = subset_dataarray_around_city(vy_da, city, buffer_degrees)

        lon2d, lat2d = np.meshgrid(rain_zoom.lon.values, rain_zoom.lat.values)

        fig, ax = plot_gridded_map(
            rain_zoom,
            title=(
                "Estimated rainfall motion field from pySTEPS optical flow\n"
                f"{city['name']} zoom, latest input: {pd.to_datetime(latest_rain.time.values)} UTC"
            ),
            cmap="Blues",
            vmin=0,
            zero_as_nan=settings.get("plot_zero_as_nan", True),
            city=city,
            figure_size=(8, 6),
            cbar_label="mm h-1",
        )

        ax.quiver(
            lon2d[::skip, ::skip],
            lat2d[::skip, ::skip],
            vx_zoom.values[::skip, ::skip],
            vy_zoom.values[::skip, ::skip],
            angles="xy",
            scale_units="xy",
            scale=1,
            width=0.003,
            zorder=3,
        )
        fig.canvas.draw_idle()


plot_motion_vectors(input_ds, velocity, skip=12)
plot_motion_vectors_city_zoom(input_ds, velocity, settings, skip=4)


# Step 12: Create a deterministic extrapolation diagnostic

Before interpreting an ensemble nowcast, it is useful to inspect a deterministic extrapolation. This shows the effect of simply moving the most recent rainfall field forward using the estimated motion vectors.

This step helps separate two concepts:

1. the estimated rainfall motion;
2. the additional uncertainty representation in the ensemble nowcast.

In [ ]:
def run_deterministic_extrapolation(precip, velocity, settings):
    """Run a deterministic pySTEPS extrapolation nowcast as a diagnostic baseline."""
    from pysteps import nowcasts

    extrapolate = nowcasts.get_method("extrapolation")
    deterministic = extrapolate(
        precip[-1, :, :],
        velocity,
        settings["n_lead_times"],
    )

    deterministic = np.where(np.isfinite(deterministic), deterministic, 0.0)
    deterministic = np.maximum(deterministic, 0.0)

    return deterministic.astype(np.float32)


deterministic_extrapolation = run_deterministic_extrapolation(precip, velocity, settings)

lead_index = 0
deterministic_plot = xr.DataArray(
    deterministic_extrapolation[lead_index],
    coords={"lat": input_ds.lat.values, "lon": input_ds.lon.values},
    dims=("lat", "lon"),
    name="precipitation_rate_deterministic_extrapolation",
    attrs={"units": "mm h-1"},
)

plot_gridded_map(
    deterministic_plot,
    title=f"Deterministic extrapolation diagnostic | lead time {lead_index + 1} h",
    cmap="Blues",
    vmin=0,
    zero_as_nan=settings.get("plot_zero_as_nan", True),
    cbar_label="mm h-1",
)


## Exercise 2: Compare persistence and deterministic extrapolation

Compare the persistence baseline and the deterministic extrapolation.

1. Which one moves the rainfall system?
2. Does the extrapolated rainfall move in the same direction as the motion vectors?
3. At what lead time does the extrapolation start to look less credible?
4. Why would an ensemble be useful in this situation?

# Step 13: Convert the nowcast to an xarray Dataset

The pySTEPS output is a NumPy array. We convert it back to a labelled xarray Dataset so that the data has clear coordinates and metadata.

The forecast axis is called `time`. This represents the **valid time** of the forecast. The original forecast lead time is stored as a coordinate called `lead_time`.

In [ ]:
def nowcast_to_xarray(forecast_array, input_ds, settings, nowcast_method):
    """Convert pySTEPS output to a labelled xarray Dataset with time as forecast axis."""
    last_input_time = pd.to_datetime(input_ds.time.values[-1])

    # Store times as timezone-naive datetime64 values that represent UTC.
    last_input_time = pd.to_datetime(last_input_time).tz_localize(None)

    lead_steps = np.arange(1, settings["n_lead_times"] + 1)
    valid_times = np.array([
        last_input_time + pd.Timedelta(minutes=settings["timestep_minutes"] * int(step))
        for step in lead_steps
    ], dtype="datetime64[ns]")

    if forecast_array.ndim == 3:
        forecast_array = forecast_array[np.newaxis, ...]
    elif forecast_array.ndim != 4:
        raise ValueError(f"Unexpected forecast shape: {forecast_array.shape}")

    ds_out = xr.Dataset(
        data_vars={
            "precipitation_rate_nowcast": (
                ["member", "time", "lat", "lon"],
                forecast_array,
                {
                    "units": "mm h-1",
                    "long_name": "Nowcast precipitation rate",
                    "nowcast_method": nowcast_method,
                    "source": "GSMaP rainfall rate with pySTEPS nowcasting",
                },
            )
        },
        coords={
            "member": np.arange(forecast_array.shape[0]),
            "time": valid_times,
            "lead_time": ("time", lead_steps),
            "lat": input_ds.lat.values,
            "lon": input_ds.lon.values,
        },
        attrs={
            "project_name": settings["project_name"],
            "country_or_region": settings["country_or_region"],
            "event_name": settings["event_name"],
            "last_observation_time_utc": str(last_input_time),
            "timestep_minutes": settings["timestep_minutes"],
            "rain_threshold_mm_h": settings["rain_threshold_mm_h"],
            "source_collection": settings["ee_collection"],
            "source_band": settings["band"],
            "time_reference": "UTC; stored as timezone-naive datetime64 for NetCDF compatibility",
        },
    )

    return ds_out


nowcast_ds = nowcast_to_xarray(forecast_array, input_ds, settings, nowcast_method)
nowcast_ds


# Step 14: Export the nowcast to NetCDF

The nowcast is saved as a NetCDF file so it can be reused later for plotting, validation, or flood-model input.

In [ ]:
last_time = pd.to_datetime(input_ds.time.values[-1]).strftime("%Y%m%dT%H%M")
nowcast_file = nowcast_folder / f"GSMaP_nowcast_{nowcast_method}_{last_time}.nc"

nowcast_ds["precipitation_rate_nowcast"].encoding.update({
    "zlib": True,
    "complevel": 4,
    "dtype": "float32",
})

nowcast_ds.to_netcdf(nowcast_file)
print(f"Saved nowcast: {nowcast_file}")


# Step 15: Inspect and visualize the nowcast results

We first plot the ensemble-mean forecast for one lead time.

The ensemble mean is useful for showing the central tendency of the forecast, but it can smooth out local extremes. For warning applications, probability maps and threshold exceedance products are often more informative.

In [ ]:
time_index = 0
forecast_mean = nowcast_ds["precipitation_rate_nowcast"].mean("member")
plot_field = forecast_mean.isel(time=time_index)

plot_gridded_map(
    plot_field,
    title=(
        f"GSMaP nowcast mean | lead time {int(plot_field.lead_time.values)} h | "
        f"valid {pd.to_datetime(plot_field.time.values)} UTC"
    ),
    cmap="Blues",
    vmin=0,
    zero_as_nan=settings.get("plot_zero_as_nan", True),
    cbar_label="mm h-1",
)


In [ ]:
max_forecast = forecast_mean.max("time")

plot_gridded_map(
    max_forecast,
    title="Maximum nowcast rainfall rate over all lead times",
    cmap="Blues",
    vmin=0,
    zero_as_nan=settings.get("plot_zero_as_nan", True),
    cbar_label="mm h-1",
)


# End of Tutorial 2

You now have a saved ensemble nowcast NetCDF in `data/gsmap/nowcast/`. **Continue with `3_Interpret_GSMaP_Nowcast.ipynb`** to:

- Compute ensemble products (exceedance probability, spread).
- Zoom in around Vientiane and Phnom Penh.
- Extract point time series and threshold-probability tables for each city.
- Compare GSMaP and the nowcast against rain-gauge observations.
- Validate the nowcast against withheld GSMaP observations.
- Export operational products to GeoTIFF.